# Hermes — Wan 2.2 video generation starter
Uses official Wan-Video/Wan2.2 code and Wan-AI/Wan2.2-TI2V-5B weights. Select GPU under Settings → Accelerator and enable Internet for initial downloads. GPU quotas and session time are limited; download outputs before the session ends.

**Hardware warning:** official TI2V-5B guidance calls for at least 24 GB VRAM. Kaggle T4x2 provides two separate 16 GB GPUs, not a single pooled 32 GB GPU, so this may run out of memory. A compatible 24 GB+ GPU is safer.

In [ ]:
import os, subprocess, torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(i, p.name, round(p.total_memory / (1024**3), 1), 'GiB VRAM')
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
%cd /kaggle/working
if not os.path.isdir('/kaggle/working/Wan2.2'):
    !git clone https://github.com/Wan-Video/Wan2.2.git
%cd /kaggle/working/Wan2.2
!pip install -q -r requirements.txt
!pip install -q 'huggingface_hub[cli]'

In [ ]:
# Download the 5B text/image-to-video model. Large download; consumes disk/time.
!hf download Wan-AI/Wan2.2-TI2V-5B --local-dir /kaggle/working/Wan2.2-TI2V-5B

In [ ]:
PROMPT = 'A cinematic slow camera push-in over a modern small business workspace, natural window light, realistic materials, gentle motion, no text or logos.'
cmd = ['python','generate.py','--task','ti2v-5B','--size','704*1280','--ckpt_dir','/kaggle/working/Wan2.2-TI2V-5B','--offload_model','True','--convert_model_dtype','--t5_cpu','--prompt',PROMPT]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Find the generated MP4 in the Wan2.2 output directory, then download it before the session ends.')

## Next steps
- Use FFmpeg to add a voiceover, licensed music, subtitles, and final 9:16 crop.
- Download the output before the Kaggle session ends.
- If generation fails with CUDA out-of-memory, do not repeatedly retry; use a supported GPU or a lower-memory implementation. Free GPU access is quota-limited and not an unlimited always-on endpoint.